In [1]:
"""
append_bbo_data.py
==================
Reads new observations from inputs.txt and outputs.txt and appends them
to the existing F1–F8 initial_inputs.npy / initial_outputs.npy files.
 
Usage
-----
    python append_bbo_data.py
 
Expected files in the same directory:
    inputs.txt              — new input rows  (one logical row per round,
                               each row contains 8 arrays, one per function)
    outputs.txt             — new output rows (same structure, 8 floats)
    f1initial_inputs.npy  … f8initial_inputs.npy
    f1initial_outputs.npy … f8initial_outputs.npy
 
Output
------
    Updated .npy files (originals backed up as f1initial_inputs_backup.npy etc.)
    append_report.txt  — human-readable summary of every change made
"""
 
import re
import sys
import shutil
import numpy as np
from pathlib import Path
from datetime import datetime
 
# ── Configuration ─────────────────────────────────────────────────────────────
N_FUNCTIONS   = 8
INPUTS_FILE   = "inputs.txt"
OUTPUTS_FILE  = "outputs.txt"
REPORT_FILE   = "append_report.txt"
 
# Expected input dimensionality for each function (used for validation)
EXPECTED_DIMS = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}
 
 
# ══════════════════════════════════════════════════════════════════════════════
# PARSING
# ══════════════════════════════════════════════════════════════════════════════
 
def read_logical_rows(text):
    """
    Split text into logical rows.  A new logical row begins whenever a line
    starts with '['.  Continuation lines (numpy array wrap) are joined on.
    Returns a list of strings, one per logical row.
    """
    rows, current = [], ''
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith('[') and current:
            rows.append(current)
            current = stripped
        else:
            current = (current + ' ' + stripped).strip() if current else stripped
    if current:
        rows.append(current)
    return rows
 
 
def parse_input_row(row_str):
    """
    Parse one logical row from inputs.txt.
    Returns a list of numpy arrays, one per function.
    """
    hits = re.findall(r'array\(\[(.*?)\]\)', row_str, re.DOTALL)
    arrays = []
    for h in hits:
        nums = [float(x) for x in re.split(r'[\s,]+', h.strip()) if x]
        arrays.append(np.array(nums))
    return arrays
 
 
def parse_output_row(row_str):
    """
    Parse one logical row from outputs.txt.
    Returns a list of floats, one per function.
    """
    hits = re.findall(r'np\.float64\((.*?)\)', row_str)
    return [float(x) for x in hits]
 
 
def load_new_data(inputs_path, outputs_path):
    """
    Load and parse both text files.
 
    Returns
    -------
    new_inputs  : list of lists — new_inputs[row][fn_idx] = np.array
    new_outputs : list of lists — new_outputs[row][fn_idx] = float
    """
    with open(inputs_path, 'r') as f:
        input_text = f.read()
    with open(outputs_path, 'r') as f:
        output_text = f.read()
 
    input_logical  = read_logical_rows(input_text)
    output_logical = read_logical_rows(output_text)
 
    if len(input_logical) != len(output_logical):
        raise ValueError(
            f"inputs.txt has {len(input_logical)} rows but "
            f"outputs.txt has {len(output_logical)} rows — they must match."
        )
 
    new_inputs  = [parse_input_row(r)  for r in input_logical]
    new_outputs = [parse_output_row(r) for r in output_logical]
 
    return new_inputs, new_outputs
 
 
# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
 
def validate(new_inputs, new_outputs, fn_idx, existing_X, existing_y):
    """
    Check that new data for function fn_idx (0-based) is dimensionally
    consistent with the existing npy arrays.  Raises ValueError on failure.
    """
    fn = fn_idx + 1
    expected_dim = existing_X.shape[1]
 
    for row_idx, (inp_row, out_row) in enumerate(zip(new_inputs, new_outputs)):
        if fn_idx >= len(inp_row):
            raise ValueError(
                f"Row {row_idx+1} of inputs.txt has only {len(inp_row)} entries "
                f"— expected at least {fn} (for F{fn})."
            )
        if fn_idx >= len(out_row):
            raise ValueError(
                f"Row {row_idx+1} of outputs.txt has only {len(out_row)} entries "
                f"— expected at least {fn} (for F{fn})."
            )
 
        x = inp_row[fn_idx]
        if x.shape[0] != expected_dim:
            raise ValueError(
                f"F{fn} row {row_idx+1}: input has {x.shape[0]} dimensions "
                f"but existing array has {expected_dim}."
            )
 
        if EXPECTED_DIMS.get(fn) and x.shape[0] != EXPECTED_DIMS[fn]:
            raise ValueError(
                f"F{fn} row {row_idx+1}: input dim {x.shape[0]} does not match "
                f"expected {EXPECTED_DIMS[fn]}."
            )
 
 
# ══════════════════════════════════════════════════════════════════════════════
# BACKUP
# ══════════════════════════════════════════════════════════════════════════════
 
def backup_file(path):
    """Copy path → path_backup.npy (overwriting any prior backup)."""
    backup = Path(str(path).replace('.npy', '_backup.npy'))
    shutil.copy2(path, backup)
    return backup
 
 
# ══════════════════════════════════════════════════════════════════════════════
# APPEND
# ══════════════════════════════════════════════════════════════════════════════
 
def append_function(fn, new_inputs, new_outputs, data_dir):
    """
    Append new observations for function fn (1-based) to its npy files.
 
    Returns a dict describing what was done, for the report.
    """
    inputs_path  = data_dir / f"f{fn}initial_inputs.npy"
    outputs_path = data_dir / f"f{fn}initial_outputs.npy"
 
    # Load existing data
    X_old = np.atleast_2d(np.load(inputs_path))
    y_old = np.asarray(np.load(outputs_path)).ravel()
 
    fn_idx = fn - 1
 
    # Validate before touching anything
    validate(new_inputs, new_outputs, fn_idx, X_old, y_old)
 
    # Collect new rows
    X_new_rows = np.array([inp_row[fn_idx] for inp_row in new_inputs])
    y_new_rows = np.array([out_row[fn_idx] for out_row in new_outputs])
 
    # Backup originals
    backup_in  = backup_file(inputs_path)
    backup_out = backup_file(outputs_path)
 
    # Concatenate
    X_updated = np.vstack([X_old, X_new_rows])
    y_updated = np.concatenate([y_old, y_new_rows])
 
    # Save
    np.save(inputs_path,  X_updated)
    np.save(outputs_path, y_updated)
 
    return {
        'fn':           fn,
        'old_rows':     X_old.shape[0],
        'new_rows':     X_new_rows.shape[0],
        'total_rows':   X_updated.shape[0],
        'dim':          X_old.shape[1],
        'backup_in':    backup_in.name,
        'backup_out':   backup_out.name,
        'X_new':        X_new_rows,
        'y_new':        y_new_rows,
    }
 
 
# ══════════════════════════════════════════════════════════════════════════════
# REPORT
# ══════════════════════════════════════════════════════════════════════════════
 
def write_report(report_path, results, timestamp):
    lines = []
    lines.append("=" * 70)
    lines.append("BBO DATA APPEND REPORT")
    lines.append(f"Run at: {timestamp}")
    lines.append("=" * 70)
 
    for r in results:
        fn = r['fn']
        lines.append(f"\nF{fn}  ({r['dim']}D)")
        lines.append(f"  Rows before : {r['old_rows']}")
        lines.append(f"  Rows added  : {r['new_rows']}")
        lines.append(f"  Rows after  : {r['total_rows']}")
        lines.append(f"  Backups     : {r['backup_in']}  /  {r['backup_out']}")
        lines.append(f"  New inputs appended:")
        for i, (x, y) in enumerate(zip(r['X_new'], r['y_new'])):
            lines.append(f"    Row {r['old_rows'] + i + 1}: "
                         f"x={np.array2string(x, precision=6, separator=', ')}  "
                         f"y={y:.10g}")
 
    lines.append("\n" + "=" * 70)
    lines.append("FINAL SHAPES")
    lines.append("=" * 70)
    lines.append(f"{'Func':<6} {'Dim':<5} {'Old rows':<10} {'Added':<8} {'New total'}")
    lines.append("-" * 40)
    for r in results:
        lines.append(f"F{r['fn']:<5} {r['dim']:<5} {r['old_rows']:<10} "
                     f"{r['new_rows']:<8} {r['total_rows']}")
 
    lines.append("\n✓ All files updated successfully.")
    text = '\n'.join(lines)
 
    with open(report_path, 'w') as f:
        f.write(text)
 
    return text
 
 
# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
 
def main(data_dir='.'):
    data_dir = Path(data_dir)
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
 
    print("=" * 70)
    print("BBO DATA APPEND UTILITY")
    print(f"Data directory : {data_dir.resolve()}")
    print(f"Started        : {timestamp}")
    print("=" * 70)
 
    # ── Load and parse text files ─────────────────────────────────────────
    inputs_path  = data_dir / INPUTS_FILE
    outputs_path = data_dir / OUTPUTS_FILE
 
    if not inputs_path.exists():
        sys.exit(f"ERROR: {inputs_path} not found.")
    if not outputs_path.exists():
        sys.exit(f"ERROR: {outputs_path} not found.")
 
    print(f"\nParsing {INPUTS_FILE} and {OUTPUTS_FILE}...")
    new_inputs, new_outputs = load_new_data(inputs_path, outputs_path)
    n_new_rows = len(new_inputs)
    print(f"  Found {n_new_rows} new observation round(s) to append.\n")
 
    # ── Process each function ─────────────────────────────────────────────
    results = []
    errors  = []
 
    for fn in range(1, N_FUNCTIONS + 1):
        inputs_npy  = data_dir / f"f{fn}initial_inputs.npy"
        outputs_npy = data_dir / f"f{fn}initial_outputs.npy"
 
        if not inputs_npy.exists() or not outputs_npy.exists():
            msg = f"F{fn}: npy files not found — skipping."
            print(f"  ⚠  {msg}")
            errors.append(msg)
            continue
 
        try:
            result = append_function(fn, new_inputs, new_outputs, data_dir)
            results.append(result)
            print(f"  ✓  F{fn}: {result['old_rows']} → {result['total_rows']} rows  "
                  f"(+{result['new_rows']} added,  {result['dim']}D)")
        except Exception as e:
            msg = f"F{fn}: FAILED — {e}"
            print(f"  ✗  {msg}")
            errors.append(msg)
 
    # ── Write report ──────────────────────────────────────────────────────
    if results:
        report_path = data_dir / REPORT_FILE
        report_text = write_report(report_path, results, timestamp)
        print(f"\nReport saved to: {report_path}")
 
    if errors:
        print("\nErrors encountered:")
        for e in errors:
            print(f"  • {e}")
        sys.exit(1)
 
    print("\n✓ All functions updated successfully.")
    return results
 
 
if __name__ == '__main__':
    # Resolve the data directory safely for both script and Jupyter cell execution.
    # __file__ is defined when run as a .py script but NOT in a Jupyter cell,
    # so we fall back to the current working directory in that case.
    try:
        script_dir = str(Path(__file__).resolve().parent)
    except NameError:
        # Running inside a Jupyter notebook cell - use the notebook working directory
        import os
        script_dir = os.getcwd()
    main(script_dir)


BBO DATA APPEND UTILITY
Data directory : /Users/mamtamehta/Documents/MyLearning/BBO13 Exercise 24
Started        : 2026-04-29 14:22:32

Parsing inputs.txt and outputs.txt...
  Found 12 new observation round(s) to append.

  ✓  F1: 10 → 22 rows  (+12 added,  2D)
  ✓  F2: 10 → 22 rows  (+12 added,  2D)
  ✓  F3: 15 → 27 rows  (+12 added,  3D)
  ✓  F4: 30 → 42 rows  (+12 added,  4D)
  ✓  F5: 20 → 32 rows  (+12 added,  4D)
  ✓  F6: 20 → 32 rows  (+12 added,  5D)
  ✓  F7: 30 → 42 rows  (+12 added,  6D)
  ✓  F8: 40 → 52 rows  (+12 added,  8D)

Report saved to: /Users/mamtamehta/Documents/MyLearning/BBO13 Exercise 24/append_report.txt

✓ All functions updated successfully.
